# 02 — ERA5 storm tracking (ground truth)

Runs PyStormTracker (Hodges algorithm + spectral filter) on the ERA5 MSL field
and extracts the Claudia ground-truth track by pre-identified `era5_track_id`.
Outputs a CSV used as truth in the track-error notebook.

In [ ]:
import pathlib

import pandas as pd
import pystormtracker as pst

In [ ]:
storm_name = "claudia"
input_file = "data/raw/msl_claudia_7d_global_2p5deg.nc"
era5_track_id = 51  # most intense Atlantic track in the Claudia window
atlantic_lat = [25.0, 65.0]
atlantic_lon = [-60.0, 10.0]
output_file = "data/interim/era5_tracks_claudia.csv"

In [ ]:
# process params
tracker = pst.HodgesTracker()
tracks = tracker.track(
    pathlib.Path(input_file),
    varname="msl",
    mode="min",
    filter=True,
)
print(f"Detected {len(tracks)} tracks")

In [ ]:
tracks_df = pd.DataFrame(
    {
        "track_id": tracks.track_ids,
        "time": pd.to_datetime(tracks.times),
        "latitude": tracks.lats,
        "longitude": tracks.lons,
        "msl_filtered": tracks.vars["msl_spectral_filtered"],
    }
)
tracks_df["longitude"] = ((tracks_df["longitude"] + 180) % 360) - 180

# If era5_track_id is 0 (auto-select), pick the most intense Atlantic track
if era5_track_id == 0:
    in_atlantic = tracks_df["latitude"].between(*atlantic_lat) & tracks_df[
        "longitude"
    ].between(*atlantic_lon)
    atlantic_ids = tracks_df.loc[in_atlantic, "track_id"].unique()
    era5_track_id = (
        tracks_df[tracks_df["track_id"].isin(atlantic_ids)]
        .groupby("track_id")["msl_filtered"]
        .min()
        .idxmin()
    )
    print(f"Auto-selected track_id: {era5_track_id}")

truth = tracks_df[tracks_df["track_id"] == era5_track_id].copy()
print(
    f"ERA5 truth track {era5_track_id}: {truth['time'].min()} → {truth['time'].max()}  ({len(truth)} steps)"
)
truth.head()

In [ ]:
output_path = pathlib.Path(output_file)
truth.to_csv(output_path, index=False)
print(f"Saved → {output_path}")